In [1]:
from steer_core.DataManager import DataManager


# Simplified imports using __init__.py
from steer_opencell_design import (
    NotchedCurrentCollector,
    AnodeFormulation,
    CathodeFormulation,
    Cathode, 
    Anode,
    Separator,
    Laminate,
    WoundJellyRoll,
    RoundMandrel,
    Tape,
    SeparatorMaterial, 
    InsulationMaterial, 
    CurrentCollectorMaterial,
    TapeMaterial, 
    AnodeMaterial,
    CathodeMaterial,
    ConductiveAdditive,
    Binder,
    __version__
)

import pandas as pd
from datetime import datetime as dt

In [2]:
db = DataManager()

In [3]:
db.remove_data(table_name='cells', condition="name = 'Cell 2'")

In [4]:
db.get_table_names()

['cells',
 'conductive_additive_materials',
 'current_collector_materials',
 'insulation_materials',
 'prismatic_container_materials',
 'separator_materials',
 'tape_materials',
 'binder_materials',
 'cathode_materials',
 'anode_materials']

In [5]:
# Get standard materials from the database

current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
insulation = InsulationMaterial.from_database("Aluminium Oxide, 95%")
separator_material = SeparatorMaterial.from_database("Polypropylene")
tape_material = TapeMaterial.from_database("Kapton")


In [6]:
# Create the current collector object

cathode_current_collector = NotchedCurrentCollector(
    material=current_collector_material,
    width=280,
    length=4800,
    thickness=12,
    tab_height=20, 
    tab_width=80, 
    tab_spacing=500,
    insulation_width=9,
    coated_tab_height=3,
)

cathode_active_material1 = CathodeMaterial.from_database("NaNiMn P2-O3 Composite")
cathode_active_material2 = CathodeMaterial.from_database("NFPP")

cathode_formulation = CathodeFormulation(
    active_materials={
        cathode_active_material1: 80,
        cathode_active_material2: 15
    },
    binders={
        binder: 2.5
    },
    conductive_additives={
        conductive_additive: 2.5
    }
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=3,
    mass_loading=15,
    insulation_material=insulation,
    insulation_thickness=3
)

In [7]:
# Create the current collector object

anode_current_collector = NotchedCurrentCollector(
    material=current_collector_material,
    width=284,
    length=4805,
    thickness=12,
    tab_height=20, 
    tab_width=80, 
    tab_spacing=500,
    insulation_width=9,
    coated_tab_height=3,
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor A)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95,},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.1,
    mass_loading=8,
    insulation_material=insulation,
    insulation_thickness=3
)

In [8]:
# make the layup

top_separator = Separator(
    material=separator_material,
    width=286,
    length=4850,
    thickness=25
)

bottom_separator = Separator(
    material=separator_material,
    width=286,
    length=4850,
    thickness=25
)

my_layup = Laminate(
    anode=my_anode,
    cathode=my_cathode,
    top_separator=top_separator,
    bottom_separator=bottom_separator
)


In [9]:
# create the jellyroll assembly

mandrel = RoundMandrel(
    diameter=5,
    length=500,
)

tape = Tape(
    material = tape_material,
    thickness=30
)

my_jellyroll = WoundJellyRoll(
    laminate=my_layup,
    mandrel=mandrel,
    tape=tape,
    additional_tape_wraps=3,
    name="Cell 2",
)


In [10]:
my_jellyroll.roll_properties

,Turns
Anode A Side Coating Turns,61.55
Anode Current Collector Turns,61.55
Anode B Side Coating Turns,61.55
Cathode A Side Coating Turns,61.28
Cathode Current Collector Turns,61.28
Cathode B Side Coating Turns,61.28
Bottom Separator Turns,63.16
Bottom Separator Inner Turns,1.43
Bottom Separator Outer Turns,0.16
Top Separator Turns,63.16


In [11]:
my_jellyroll.get_spiral_plot(height=1300, width=1300).show()

In [12]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_jellyroll.name],
    'object': [my_jellyroll.serialize()],
    'form_factor': ['Cylindrical'],
    'internal_construction': ['Wound'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [13]:
db.get_data(table_name='cells')

,name,object,form_factor,internal_construction,date,version,reference
0,Cell 4,gASVkQkBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-18 13:33:19,0.4.7,Na/Na+
1,HiNa-NaCR32140-MP10,gASV9gwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-18 13:33:21,0.4.7,Na/Na+
2,QNAS-S40160NL,gASV9gwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-18 13:33:23,0.4.7,Na/Na+
3,QNAS-S40160RL,gASV9gwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-18 13:33:25,0.4.7,Na/Na+
4,UniGrid-NCO32140,gASVPSEBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-18 13:33:27,0.4.7,Na/Na+
5,Vendor D NFPP,gASVGwABAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-11-18 13:33:30,0.4.7,Na/Na+
6,Vendor E NFPP,gASVKQABAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-11-18 13:33:31,0.4.7,Na/Na+
7,Vendor G NFM,gASV9gwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-18 13:33:33,0.4.7,Na/Na+
8,Vendor G NFPP,gASV8iYBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-18 13:33:36,0.4.7,Na/Na+
9,CBAK-32140NS,gASVOw0AAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-12-01 14:38:24,0.4.9,Na/Na+
